# videoscore クイックスタート（Python）

VideoScore 中間構造を `videoscore.model`（pydantic）で読み書き・検証する基本操作。

事前に型パッケージを入れておく:

```bash
pip install -e ..            # このリポジトリの python/ で開発インストール
# もしくは git 経由:
# pip install "git+https://github.com/Nu424/videoscore-format.git#subdirectory=python"
```

仕様の一次ソースは `../../documents/intermediate-structure-guideline.md`。

## 0. import

In [ ]:
import json
from pydantic import ValidationError
from videoscore.model import (
    VideoScore, Scene, AudioElement, TelopElement, VideoElement,
    StyleCatalog, validate_styles,
)

## 1. JSON から読み込む（パース＋検証が同時に走る）

`model_validate_json` は不正な構造（未知キー、role の enum 外、`start` に `auto` 等）をその場で弾く。

In [ ]:
raw = """
{
  "meta": { "title": "サンプル", "fps": 30, "size": [1920, 1080] },
  "scenes": [
    {
      "id": "s1",
      "duration": { "ref": "audio.end" },
      "audio": [
        { "id": "v1", "role": "voice", "source": "tts://まずは結論から", "t": [0, "auto"] }
      ],
      "telop": [
        { "t": [0, { "ref": "v1.end" }], "text": "まずは結論から", "style": "tone.emphasis" }
      ],
      "video": [
        { "source": "broll.mp4", "in": 0, "out": 8, "t": [0, "auto"] }
      ]
    }
  ]
}
"""

doc = VideoScore.model_validate_json(raw)
print("scenes:", len(doc.scenes), "| s1.duration:", doc.scenes[0].duration)

## 2. 構造にアクセスする（時間語彙は脱糖されず素のまま）

`t` の `"auto"` / `"after"` / `{ref}` はそのまま保持される。脱糖・時間解決は解決スクリプト（将来の `videoscore.resolve`）の責務。

In [ ]:
v1 = doc.scenes[0].audio[0]
print("audio[0]:", v1.role, v1.source, "| t =", v1.t)

# telop の end はレーン跨ぎ同期の参照（RefObject）
print("telop[0].t[1]:", doc.scenes[0].telop[0].t[1])

## 3. プログラムから組み立てる

`in` は Python の予約語なので、フィールド名は `in_`（出力時は `"in"` に戻る）。

In [ ]:
scene2 = Scene(
    id="s2",
    duration={"ref": "video.end"},
    video=[
        VideoElement(source="a.mp4", in_=0, out=4, t=(0, "auto")),
        VideoElement(source="b.mp4", in_=10, out=13, t=("after", "auto")),
    ],
    telop=[TelopElement(t=(1.0, 6.0), text="カットを跨ぐテロップ", style="telop.default")],
)
doc.scenes.append(scene2)
print("appended:", [s.id for s in doc.scenes])

## 4. 最小 JSON として出力する

`to_json_dict()` は `None` フィールドと空レーンを省き、`in` 等はエイリアスで出す（入力に近い素の形）。

In [ ]:
out = doc.to_json_dict()
# s2 は audio/overlay を持たない → 空レーンは出力されない
print("s2 keys:", list(out["scenes"][1].keys()))
print(json.dumps(out["scenes"][1], ensure_ascii=False, indent=2))

## 5. スタイルをカタログと突き合わせる（§7 enum / appliesTo）

`validate_styles` は例外を投げず、問題のリストを返す（空なら OK）。

In [ ]:
catalog = StyleCatalog.model_validate({
    "styles": {
        "tone.emphasis": {"intent": "決め台詞", "feeling": "強い", "appliesTo": ["telop"]},
        "telop.default": {"intent": "通常字幕", "feeling": "中立", "appliesTo": ["telop"]},
    }
})
print("issues (OK):", validate_styles(doc, catalog))

# telop 専用の印を audio に付けると appliesTo 違反になる
doc.scenes[0].audio[0].style = "tone.emphasis"
for issue in validate_styles(doc, catalog):
    print(issue)

## 6. 検証エラーの例：`start` に `"auto"` は不可

循環防止の鉄則「start は具体 or 参照、派生してよいのは end だけ」を**型レベル**で表現している。

In [ ]:
try:
    TelopElement(t=("auto", 3.0), text="x")
except ValidationError as e:
    print("rejected as expected:")
    print(e)

## 7. （おまけ）JSON Schema を取り出す

pydantic が Single Source of Truth。`videoscore-gen-schema` はこれを `schema/*.json` に書き出し、そこから TypeScript 型を生成する。

In [ ]:
schema = VideoScore.model_json_schema()
print("defs:", list(schema["$defs"].keys()))